In [1]:
import argparse
import os
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader
from model.dataset import BagsDataset, custom_collate_fn
from model.model import MIL
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm import tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
all_genes = pd.read_csv('human.csv')
all_genes = all_genes['Gene'].values

In [5]:
dataset = load_datasets(data_dir='/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE', immune_cell='tcell',radius=200,)

Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:01<00:00, 2888.61it/s]


Total bags created: 3858
Average instances per bag: 8


Creating Bags with radius 200: 100%|██████████████████████████| 3858/3858 [00:01<00:00, 3130.13it/s]

Total bags created: 3858
Average instances per bag: 8


In [6]:
dataset = dataset[0] + dataset[1]

In [7]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)


In [8]:
model = MIL(all_genes)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)
model.to(device)

MIL(
  (distance): Distance(
    (sparsemax): Sparsemax()
  )
  (gene_expression): Gene_expression(
    (softmax): Softmax(dim=-1)
  )
  (immunogenicity): Immunogenicity()
)

In [9]:
num_epochs = 5

for epoch in range(num_epochs):
    model.train() 
    running_loss = 0.0
    
    with tqdm(dataloader, unit="batch") as tepoch:
        for i, (distances, gene_expressions, label, core_idx, current_genes) in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")

            optimizer.zero_grad()
            output = model(distances, gene_expressions,current_genes)
            #print(output)
            #print(label)
            loss = criterion(output, label)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            tepoch.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')


Epoch 1/5:   0%|          | 0/7716 [00:00<?, ?batch/s]


NameError: name 'F' is not defined